# SpecDist — Colab A100 Quickstart

**Distil Qwen3-0.6B from Qwen3-8B on an A100 (Colab Pro/Pro+).**

| CONFIG | Teacher | Steps | Time | Purpose |
|--------|---------|-------|------|---------|
| `a100` | 8B (BF16) | 2000 | ~2-3 h | Paper-quality results |

Set `CONFIG = "a100"` in Cell 0 and **Runtime → Run all** (Ctrl+F9).  
For free T4 use `colab_quickstart.ipynb` (`colab_lite` or `colab`).

| Cell | What it does | Run time |
|------|-------------|----------|
| 0. Bootstrap | **One-shot: setup + auth + run** | ~3 min to start |
| 1. Setup | Mount Drive, clone repo, install deps | ~3 min |
| 2. Auth | W&B + HuggingFace | ~30 s |
| 3. Run | Full pipeline (train → merge → eval) | ~2-3 h |
| 4. Resume | After session death — skips completed steps | ~1 min + rest |
| 5. Monitor | State + log tail (auto-refresh option) | instant |
| 6. Dashboard | Live results dashboard in a browser tab | ~10 s |
| 7. Tree loss (single) | Train one tree loss | ~10-17 min |
| 8. Tree losses (all) | Train all 9 tree losses sequentially | ~2-3 h |

### Requires Colab Pro or Pro+
Runtime → Change runtime type → **A100 GPU**  
(Free tier gives T4 only — use `colab_quickstart.ipynb` for that.)

### Why A100?
- 8B teacher loads as plain BF16 — no 4-bit needed, no OOM risk
- 40 GB VRAM: 1.2 GB draft + 16 GB teacher + activations = ~19 GB used
- `torch.compile` enabled: +10-30% step speed
- LoRA r=16 (vs r=8 on T4): better gap closure for 8B→0.6B distillation

### Prerequisites
1. **A100 runtime**: Runtime → Change runtime type → **A100 GPU**
2. **Colab Secrets** (left sidebar → 🔑 icon):
   - `GITHUB_TOKEN` — **required** (private repo).  
     Create a classic PAT with `repo` scope at https://github.com/settings/tokens
   - `WANDB_API_KEY` — from https://wandb.ai/authorize
   - `HF_TOKEN` — from https://huggingface.co/settings/tokens
3. Google Drive (mounted automatically by Cell 0 or Cell 1)

> First run downloads Qwen3-8B (~16 GB, ~15 min).  
> Subsequent runs use the cached copy in `/root/.cache/huggingface/`.

In [ ]:
# =============================================================================
# Cell 0 — ONE-SHOT BOOTSTRAP (A100)
# Edit CONFIG + flags below, then Runtime → Run all (Ctrl+F9).
# =============================================================================

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"

# ── Edit these ────────────────────────────────────────────────────────────────────────────
CONFIG      = "a100"   # a100  (8B teacher, 2000 steps)
SMOKE       = False          # True = 10-step crash check
BACKGROUND  = False          # True = background; monitor with Cell 5
LOSSES      = None           # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

def _prereq_token(name):
    """Read a Colab secret before deploy_utils is available (needed for git clone)."""
    try:
        from google.colab import userdata; v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh
else: print("⚠ GITHUB_TOKEN not set — clone will fail for a private repo.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Repo updated")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, DRIVE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          mount_drive=True, warn_vram_below_gb=30.0)
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR,
             smoke=SMOKE, losses=LOSSES,
             background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — Setup  (alternative to Cell 0: run Cells 1 → 2 → 3 separately)
# =============================================================================
import os, subprocess, sys

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)
print(f"✓ Artifacts persist at: {DRIVE_ROOT}")

# Minimal bootstrap: deploy_utils (which has _auth_repo_url) needs the clone first.
try: from google.colab import userdata; gh = userdata.get("GITHUB_TOKEN") or ""
except Exception: gh = os.environ.get("GITHUB_TOKEN", "")
if not gh:
    print("⚠ GITHUB_TOKEN not set — clone will fail for a private repo.")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated")

GBV_DIR = f"{REPO_DIR}/gbv-research"
os.chdir(GBV_DIR)

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import install_deps, fetch_training_data
install_deps(GBV_DIR)
fetch_training_data(GBV_DIR)
print("--- Setup done. Run Cell 2. ---")

In [ ]:
# =============================================================================
# Cell 2 — Authenticate  (W&B + HuggingFace + GPU check)
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import auth_wandb, auth_hf, check_gpu

auth_wandb()
auth_hf()
check_gpu(warn_below_gb=30.0)   # warns if not A100
print("--- Auth done. Run Cell 3. ---")

In [ ]:
# =============================================================================
# Cell 3 — Run pipeline  (flat losses: kl, jsd, l1, ebe, …)
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, fetch_training_data

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ────────────────────────────────────────────────────────────────
CONFIG      = "a100"  # a100 (8B teacher, 2000 steps)
SMOKE       = False
BACKGROUND  = False
LOSSES      = None
# ─────────────────────────────────────────────────────────────────────────────

fetch_training_data(GBV_DIR)
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSSES, background=BACKGROUND)

In [ ]:
# =============================================================================
# Cell 4 — Resume after session death (A100)
# Re-mounts Drive, re-clones if needed, re-auths, resumes pipeline.
# The pipeline skips already-completed steps automatically.
# =============================================================================

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
CONFIG     = "a100"   # must match what was used in Cell 3

import os, subprocess, sys

def _prereq_token(name):
    """Read a Colab secret before deploy_utils is available (needed for git clone)."""
    try:
        from google.colab import userdata; v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh
else: print("⚠ GITHUB_TOKEN not set — clone will fail for a private repo.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Repo updated")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, DRIVE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          mount_drive=True, warn_vram_below_gb=30.0)
print(f"\nResuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 6 — Monitor  (safe to run any time, including while pipeline runs)
# AUTO_REFRESH = True → live tail; interrupt cell to stop.
#
# CONFIG must match the PROFILE used in Cell 7 or Cell 8.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import monitor

DRIVE_ROOT   = "/content/drive/MyDrive/specdist"
# ── Must match Cell 7/8 PROFILE ─────────────────────────────────────────────
CONFIG       = "profiles/a100_tree_losses"
# Other options:
#   "a100"                            # legacy flat-loss runs on A100
# ────────────────────────────────────────────────────────────────────────────
AUTO_REFRESH = False
REFRESH_SECS = 20

monitor(DRIVE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 6 — Dashboard
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import start_dashboard

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

start_dashboard(DRIVE_ROOT, GBV_DIR)

---

## Tree-Structured Losses (Track B) — A100

Use **Cell 7** for one loss at a time, **Cell 8** for all 9.

| PROFILE | Teacher | Steps | Time/loss |
|---------|---------|-------|-----------|
| `profiles/a100_tree_losses` | 8B | 2000 | ~10-17 min |

All 9 tree losses — see `colab_quickstart.ipynb` for the full table.  
Evaluates with `bv`, `gbv`, `traversal` only (OT verifiers out-of-distribution for tree-trained models).

**Prerequisite:** Cells 1 + 2 must have run (or Cell 0).

In [ ]:
# =============================================================================
# Cell 7 — Train a single tree loss (A100)
# background=True: cell returns immediately — run Cell 6 (monitor) in parallel.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, show_profile, keep_alive

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ──────────────────────────────────────────────────────────────────────────────
LOSS    = "kl_tree"
PROFILE = "profiles/a100_tree_losses"   # 8B teacher, 2000 steps
SMOKE   = False
# ─────────────────────────────────────────────────────────────────────────────────

keep_alive()
show_profile(PROFILE, GBV_DIR)
print(f"  Loss: {LOSS}")

# background=True: log-tailing thread streams output; run Cell 6 (monitor) anytime.
run_pipeline(PROFILE, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSS, background=True)

In [ ]:
# =============================================================================
# Cell 8 — Train ALL tree losses sequentially (A100)
# Re-run to resume — completed steps are always skipped.
# Runs in a background thread — Cell 6 (monitor) and dashboard stay available.
# =============================================================================
import sys, threading
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, show_profile, keep_alive

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ──────────────────────────────────────────────────────────────────────────────
PROFILE = "profiles/a100_tree_losses"
LOSSES_TO_RUN = [
    "kl_tree", "rev_kl_tree", "jsd_tree",
    "bv_tree", "gbv_tree", "traversal_tree",
    "ebe_tree",
    "online_kl_tree", "online_ebe_tree",
]
SMOKE = False
# ─────────────────────────────────────────────────────────────────────────────────

keep_alive()
show_profile(PROFILE, GBV_DIR)
print(f"  Losses: {len(LOSSES_TO_RUN)} × tree  |  resume-safe\n")

def _run_all_losses():
    for i, loss in enumerate(LOSSES_TO_RUN):
        print(f"{'='*50}\n[{i+1}/{len(LOSSES_TO_RUN)}] {loss}\n{'='*50}")
        # background=True streams log output; proc.wait() keeps losses sequential
        # without blocking the kernel — Cell 6 (monitor) can run in parallel.
        proc = run_pipeline(PROFILE, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=loss,
                            background=True)
        proc.wait()
        rc = proc.returncode
        if rc != 0:
            print(f"  [FAIL] {loss} exited {rc} — re-run to resume.")
            break
    else:
        print("\n✓ All tree losses complete — run Cell 6 for the dashboard.")

threading.Thread(target=_run_all_losses, daemon=True).start()
print("Losses running in background — Cell 6 (monitor) and dashboard are available now.")

In [ ]:
# =============================================================================
# Cell 4 — Train ONE loss  (Colab — one loss at a time, --skip_existing handles resume)
# Change LOSS below and re-run.  --skip_existing skips completed steps.
# This is the standard per-session workflow on Kaggle:
#   Session 1: LOSS = 'kl'        Session 2: LOSS = 'rev_kl' etc.
# On persistent machines (A100): change LOSS each time you want the next loss.
# =============================================================================
import sys
sys.path.insert(0, '/content/Distill-Spec-Research/gbv-research/deploy')
from deploy_utils import run_pipeline

DRIVE_ROOT = '/content/drive/MyDrive/specdist'
GBV_DIR    = '/content/Distill-Spec-Research/gbv-research'

# ── Edit this each session ───────────────────────────────────────────────────
LOSS   = 'kl'      # kl | rev_kl | jsd | l1 | ebe | ebe_single
#                   # kl_tree | bv_tree | gbv_tree | traversal_tree
#                   # rev_kl_tree | jsd_tree | naive_tree | nss_tree
#                   # specinfer_tree | spectr_tree | khisti_tree
CONFIG = 'a100'   # must match the CONFIG used in Cell 0
SMOKE  = False         # True = 10-step crash check
# ─────────────────────────────────────────────────────────────────────────────

# Does: baseline eval (once, --skip_existing) → train LOSS → merge → eval GSM8K
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSS, background=False)


In [ ]:
# =============================================================================
# Cell 5 — Eval ONE trained model  (no retraining)
# Use when the checkpoint is already done but you want to re-run eval
# (different K, different n_prompts, extra verifier modes, etc.)
# Runs evaluate.py directly — bypasses the full pipeline orchestration.
# =============================================================================
import sys, os
sys.path.insert(0, '/content/Distill-Spec-Research/gbv-research')
GBV_DIR    = '/content/Distill-Spec-Research/gbv-research'
DRIVE_ROOT = '/content/drive/MyDrive/specdist'

# ── Edit these ───────────────────────────────────────────────────────────────
LOSS    = 'kl'           # which trained model to evaluate
CONFIG  = 'a100'     # must match training config
MODES   = 'alpha,bv,gbv,traversal,specinfer,naive'  # verifier modes
K       = '3'            # draft paths
TEMP    = '1.0'          # sampling temperature
N       = '100'          # number of eval prompts
DATASET = 'gsm8k'        # gsm8k | humaneval | math500 | mtbench | alpaca
# ─────────────────────────────────────────────────────────────────────────────

import subprocess
ckpt = os.path.join(DRIVE_ROOT, 'checkpoints', f'{LOSS.replace("_","_")}-gsm8k_merged')
if not os.path.isdir(ckpt):
    ckpt = os.path.join(DRIVE_ROOT, 'checkpoints', f'{LOSS}-gsm8k_merged')
    print(f'Trying: {ckpt}')
if not os.path.isdir(ckpt):
    raise FileNotFoundError(f'Merged checkpoint not found: {ckpt}\n'
                            'Run Cell 4 first to train + merge the model.')

# Import the target model path from the running config
import yaml
cfg_path = os.path.join(GBV_DIR, 'orchestration', 'configs', f'{CONFIG}.yaml')
target = yaml.safe_load(open(cfg_path))['models']['target']

cmd = [
    'python', os.path.join(GBV_DIR, 'orchestration', 'evaluate.py'),
    '--student', ckpt,
    '--teacher', target,
    '--student_label', LOSS,
    '--datasets', DATASET,
    '--modes', MODES,
    '--K', K, '--temperature', TEMP, '--n', N,
    '--skip_fetch', '--skip_existing',
    '--storage_root', DRIVE_ROOT,
    '--loss_name', LOSS,
]
print('Running:', ' '.join(cmd[-10:]))
result = subprocess.run(cmd, cwd=GBV_DIR)
print('Done — refresh dashboard to see results.')
